# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets, merge_xai_results
from src.model.functions import save_model, load_model, get_split_data
from src.xai.functions import get_train_iterations_shap, compute_xai_metrics_from_shap
from src.graph.functions import plot_model_boxplot, plot_imbalance_boxplot_grid

try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd
from pathlib import Path

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
#datasets_to_run = ["australian", "german", "japan", "kgccdea", "taiwan", "th02"] #kggmsc #taiwan
datasets_to_run = ["australian", "german", "japan", "kgccdea", "th02"] #kggmsc #taiwan
datasets_to_run = ["australian", "japan"]
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [10, 15, 20, 25, 30, 35, 40, 45, 50]
imbalance_levels = [10, 15, 20, 25]
n_iterations = 100
percentage = 90

# list of estimators understood by functions.get_model
model_choices = ["RandomForest", "XGBoost", "LightGBM", "BoostDT", "BoostSGB", "BagNN", "AdaBoost-DT", "ANN"] #, "SVM-Rbf"
model_choices = ["LightGBM"]
product = "default"

## Create the imbalance datasets

In [3]:
for dataset_name in datasets_to_run:
    dataset_path = data_dir / "input" / f"{dataset_name}.csv"
    # Check if file exists and read it using the helper from src.csv_config
    if dataset_path.exists():
        # Use the dataset-specific defaults (german uses tab separator)
        create_imbalanced_datasets(
            dataset_name=dataset_name,
            data_dir=data_dir,
            imbalance_levels=imbalance_levels,  # <-- modify freely
            target_col="target",
            )

Original dataset: australian
Total samples: 690
Positive samples: 307
Negative samples: 383
Positive rate: 0.4449

australian_10.csv | Total=420 | Pos=42 | Neg=378 | Rate=0.1000 | Target=0.1000
australian_15.csv | Total=440 | Pos=66 | Neg=374 | Rate=0.1500 | Target=0.1500
australian_20.csv | Total=475 | Pos=95 | Neg=380 | Rate=0.2000 | Target=0.2000
australian_25.csv | Total=508 | Pos=127 | Neg=381 | Rate=0.2500 | Target=0.2500

Finished generating imbalanced datasets.
Original dataset: japan
Total samples: 690
Positive samples: 307
Negative samples: 383
Positive rate: 0.4449

japan_10.csv | Total=420 | Pos=42 | Neg=378 | Rate=0.1000 | Target=0.1000
japan_15.csv | Total=440 | Pos=66 | Neg=374 | Rate=0.1500 | Target=0.1500
japan_20.csv | Total=475 | Pos=95 | Neg=380 | Rate=0.2000 | Target=0.2000
japan_25.csv | Total=508 | Pos=127 | Neg=381 | Rate=0.2500 | Target=0.2500

Finished generating imbalanced datasets.


## Train the models

In [4]:
results = []

for dataset_name in datasets_to_run:
    # variants for this dataset
    dataset_variants = [dataset_name] + [f"{dataset_name}_{pct}" for pct in imbalance_levels]

    # ensure output dir exists
    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:
        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Read each dataset variant (uses your resolver logic in csv_config.py)
        df = pd.read_csv(str(dataset_path))

        # Basic safety checks
        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found. Columns={list(df.columns)[:10]}...")
            continue

        # Quick dataset stats
        n_total = len(df)
        # Make sure sum/mean work even if target is int-like
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        # Prepare column lists for this df
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Optionally skip if only one class present (prevents stratify crash)
        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present (pos={n_pos}, neg={n_neg}).")
            continue

        for model_choice in model_choices:
            print(f"\n=== training {model_choice} on {dataset_variant} ===")

            pipeline, best_params, best_score = get_train_model(
                data=df,
                feature_cols=feature_cols,
                numerical_cols=numerical_cols,
                categorical_cols=categorical_cols,
                ids_cols=ids_cols,
                target_cols=target_cols,
                exclude_cols=exclude_cols,
                product=product,
                algorithm=model_choice,
                test_mode=True,  # set False for full search
            )

            print("Best hyperparameters:", best_params)
            print("Best CV score:", best_score)

            model_path = out_dir / "model" /f"{dataset_variant}_{model_choice}.pkl"
            save_model(str(model_path), pipeline, environment="local")
            print(f"trained pipeline saved to {model_path}")

            results.append(
                {
                    "base_dataset": dataset_name,
                    "dataset_variant": dataset_variant,
                    "model": model_choice,
                    "n_total": n_total,
                    "n_pos": n_pos,
                    "pos_rate": pos_rate,
                    "params": best_params,
                    "score": best_score,
                }
            )

results_df = pd.DataFrame(results)
results_df


##########################################################################################
BASE DATASET: australian
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/australian/australian.csv

DATASET: australian_10 | n=420 | pos=42 | pos_rate=0.1000

=== training LightGBM on australian_10 ===
Search space for algorithm=LightGBM, product=default, test_mode=True: {'model__n_estimators': Integer(low=50, high=300, prior='uniform', transform='identity'), 'model__max_depth': Integer(low=3, high=6, prior='uniform', transform='identity'), 'model__learning_rate': Real(low=0.01, high=0.2, prior='log-uniform', transform='identity'), 'model__scale_pos_weight': Integer(low=1, high=10, prior='uniform', transform='identity'), 'model__num_leaves': Integer(low=20, high=150, prior='uniform', transform='identity'), 'model__min_child_samples': Integer(low=20, high=150, 

,base_dataset,dataset_variant,model,n_total,n_pos,pos_rate,params,score
0,australian,australian_10,LightGBM,420,42,0.10,{'model__bagging_fraction': 0.9293468848796316...,0.098639
1,australian,australian_15,LightGBM,440,66,0.15,{'model__bagging_fraction': 0.9293468848796316...,0.149351
2,australian,australian_20,LightGBM,475,95,0.20,{'model__bagging_fraction': 0.9293468848796316...,0.198795
3,australian,australian_25,LightGBM,508,127,0.25,{'model__bagging_fraction': 0.9293468848796316...,0.250698
4,japan,japan_10,LightGBM,420,42,0.10,{'model__bagging_fraction': 0.9293468848796316...,0.098639
5,japan,japan_15,LightGBM,440,66,0.15,{'model__bagging_fraction': 0.9293468848796316...,0.149351
6,japan,japan_20,LightGBM,475,95,0.20,{'model__bagging_fraction': 0.9293468848796316...,0.198795
7,japan,japan_25,LightGBM,508,127,0.25,{'model__bagging_fraction': 0.9293468848796316...,0.250698


## SHAP iterations

In [5]:
# -------------------------
# Loop XAI iterations
# -------------------------
for dataset_name in datasets_to_run:

    dataset_variants = [dataset_name] + [f"{dataset_name}_{lvl}" for lvl in imbalance_levels]

    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:

        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Same reader used during training
        path = str(dataset_path)
        df = pd.read_csv(str(path))

        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found.")
            continue

        n_total = len(df)
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present.")
            continue

        # Same column preparation as training
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Data split
        x_train, x_test, y_train, y_test, numerical, categorical = get_split_data(
            data=df,
            feature_cols=feature_cols,
            numerical_cols=numerical_cols,
            categorical_cols=categorical_cols,
            ids_cols=ids_cols,
            target_cols=target_cols,
            exclude_cols=exclude_cols,
            test_size=0.5,
            random_state=123,
        )

        for model_choice in model_choices:

            model_path = out_dir / "model" /f"{dataset_variant}_{model_choice}.pkl"

            if not model_path.exists():
                print(f"[SKIP] Model not found: {model_path}")
                continue

            print(f"Running XAI iterations for {dataset_variant} | {model_choice}")

            pipeline = load_model(str(model_path))

            df_shap = get_train_iterations_shap(
                pipeline=pipeline,
                x_train=x_train,
                x_test=x_test,
                y_train=y_train,
                n_iterations=n_iterations,
                percentage=percentage,
            )
            # ----------------------------
            # Create output folder
            # ----------------------------
            result_dir = out_dir / "xai" / f"{model_choice}" / f"{dataset_variant}" / "shap_values"
            result_dir.mkdir(parents=True, exist_ok=True)

            # ----------------------------
            # Save outputs
            # ----------------------------
            df_shap.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_shap_values.csv",
                index=False
            )

            print(f"[OK] Saved results to {result_dir}")


##########################################################################################
BASE DATASET: australian
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/australian/australian.csv

DATASET: australian_10 | n=420 | pos=42 | pos_rate=0.1000
Running XAI iterations for australian_10 | LightGBM
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/model/australian_10_LightGBM.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/LightGBM/australian_10/shap_values

DATASET: australian_15 | n=440 | pos=66 | pos_rate=0.1500
Running XAI iterations for australian_15 | LightGBM
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/model/australian_15_LightGBM.pkl
[OK] Sa